# 수치해석 03. 수치선형대수

## 학습 목표
- 핵심 정의와 정리를 자신의 말로 설명한다.
- 기본 개념 예제를 손계산으로 확인한다.
- Python 코드와 시각화를 통해 직관을 검산한다.

## 핵심 개념
- 조건수
- 반복법
- 선형시스템
- 안정성

## 기본 개념 예제
조건수가 큰 행렬에서 해가 얼마나 흔들리는지 본다.

## 실제 응용 예제
센서 보정 행렬이 불안정한지 진단한다.

## 이론 정리

### 정의와 관점
- 수치해석은 연속적인 수학 문제를 유한한 계산 절차로 바꾸는 학문이다.
- 오차는 절단오차, 반올림오차, 모델오차, 데이터오차로 나누어 생각한다.
- 안정성은 입력이나 중간 계산의 작은 오차가 결과에서 얼마나 증폭되는지를 묻는다.
- 조건수는 문제 자체가 얼마나 민감한지, 안정성은 알고리즘이 그 민감도를 얼마나 키우는지와 관련된다.

### 핵심 명제와 정리
- Newton 방법은 적절한 조건에서 근 근처에서 매우 빠르게 수렴하지만 초기값에 민감하다.
- 보간 오차는 노드 선택과 함수의 고계도함수에 의존한다.
- 수치선형대수에서는 조건수와 잔차가 해의 신뢰도를 판단하는 핵심 지표다.
- 일관성, 안정성, 수렴성은 수치 미분방정식 해법의 기본 축이다.

### 계산과 학습 절차
- 문제의 스케일, 조건수, 허용 오차를 먼저 정한다.
- 알고리즘을 적용한 뒤 잔차, 반복횟수, 오차 추정량을 함께 기록한다.
- 격자나 차수를 바꿔 결과가 수렴하는지 확인한다.
- 가능하면 해석해가 있는 테스트 문제로 구현을 먼저 검증한다.

### 자주 생기는 오해
- 잔차가 작아도 조건수가 크면 실제 해 오차가 클 수 있다.
- 차수를 무작정 높이면 Runge 현상처럼 오히려 근사가 나빠질 수 있다.
- 부동소수점 계산에서는 너무 작은 간격이 항상 더 정확한 것은 아니다.

### 증명으로 연결하기
- 수렴성 증명은 국소오차와 안정성 추정을 결합해 전역오차를 제어한다.
- 반복법 분석은 오차 전달 행렬의 스펙트럼 반경을 본다.
- 보간과 근사 증명은 나머지항을 명시해 오차가 어디서 오는지 추적한다.

### 이 챕터에서 꼭 확인할 질문
- 기본 개념 예제 "조건수가 큰 행렬에서 해가 얼마나 흔들리는지 본다."에서 실제로 사용한 정의는 무엇인가?
- 응용 예제 "센서 보정 행렬이 불안정한지 진단한다."에서 어떤 가정이 현실을 단순화하고 있는가?
- 코드가 연속 대상을 이산화한다면, 격자나 표본 수를 바꾸어도 결론이 유지되는가?
- 손계산 가능한 작은 사례와 노트북 결과가 같은 결론을 주는가?

## 0. 실행 준비

아래 셀은 프로젝트 루트의 `common/math_viz.py`를 찾아서 현재 챕터의 출력 폴더를 자동으로 설정합니다. Jupyter Lab을 프로젝트 루트에서 열면 가장 안정적으로 동작합니다.

In [ ]:
from pathlib import Path
import sys
from IPython.display import Image, display


CHAPTER_RELATIVE_DIR = Path("3학년_순수수학과_응용수학의_분화/08_수치해석/ch03_수치선형대수")


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "common" / "math_viz.py").exists():
            return candidate
    raise RuntimeError("common/math_viz.py를 찾지 못했습니다. Jupyter Lab을 프로젝트 루트에서 열어 주세요.")


ROOT = find_project_root(Path.cwd().resolve())
NOTEBOOK_DIR = ROOT / CHAPTER_RELATIVE_DIR
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
sys.path.insert(0, str(ROOT / "common"))

from math_viz import PROFILES, run_profile

PROFILE = "numerical_linear"
TITLE = "수치해석 - 수치선형대수"
CONCEPT_EXAMPLE = "조건수가 큰 행렬에서 해가 얼마나 흔들리는지 본다."
APPLICATION_EXAMPLE = "센서 보정 행렬이 불안정한지 진단한다."

print("project root:", ROOT)
print("chapter dir:", NOTEBOOK_DIR)
print("profile:", PROFILE)

## 1. 이번 챕터의 시각화 코드 읽기

먼저 실제로 실행될 함수를 확인합니다. 코드를 읽으면서 입력값, 이산화 방식, 그래프가 의미하는 수학적 대상을 표시해 보세요.

In [ ]:
import inspect

print(inspect.getsource(PROFILES[PROFILE]))

## 2. 실행하고 결과 확인하기

아래 셀을 실행하면 `outputs/visualization.png`가 생성되고, 노트북 안에도 바로 표시됩니다.

In [ ]:
run_profile(
    profile=PROFILE,
    title=TITLE,
    concept=CONCEPT_EXAMPLE,
    application=APPLICATION_EXAMPLE,
    output_dir=OUTPUT_DIR,
)

display(Image(filename=str(OUTPUT_DIR / "visualization.png")))

## 3. 변형 실험

- 표본 수, 격자 크기, 초기값, 학습률, 경계조건 중 하나를 바꿔 보세요.
- 그림이 안정적으로 유지되는 범위와 결론이 바뀌는 범위를 나누어 적어 보세요.
- 손계산 가능한 작은 예제를 만들어 코드 결과와 비교해 보세요.

In [ ]:
# 여기에 자신만의 변형 실험을 작성하세요.
# 예: common/math_viz.py에서 위에 출력된 함수의 파라미터를 복사해 와서
#     표본 수, 구간, 초기값 등을 바꾼 뒤 다시 그려 볼 수 있습니다.
